# Herramienta de análisis de imágenes biomédicas (RMN)

Este cuaderno te guiará paso a paso para usar el modelo de Inteligencia Artificial que has entrenado en el notebook de Entrenamiento.

⚠️ No necesitas conocimientos de programación.

Solo sigue las instrucciones y ejecuta las celdas en orden.

# Requisitos

Antes de empezar, debes tener en Google Drive:


*   El modelo entrenado (.h5)
*   La resonancia a analizar (.nii / .nii.gz)



# Instalación de librerias necesarias

**Este paso solo debe ejecutarse una vez.**

Si ya lo ejecutaste anteriormente, puedes saltarlo.

In [ ]:
#@title 🔧 Instalación de dependencias

!pip install plotly

# Paso 1 - Carga de modelo e imagen

Pulsa el botón y selecciona las rutas donde tienes el modelo y la imagen.

El sistema comprobará automáticamente:


*   Que las rutas existen.
*   Los tipos de archivo son correctos.

No necesitas hacer nada más.



In [ ]:
#@title ☁️ Carga de requerimientos


# -----------------------------
# MONTANDO GOOGLE DRIVE
# -----------------------------
import os
from google.colab import drive
from rich import print
from rich.panel import Panel

print(Panel("Montando Google Drive...", style="yellow"))
drive.mount('/content/drive')

# -----------------------------
# INSTRUCCIONES
# -----------------------------
print(Panel(
    "PASO 1 · Cargar el modelo y la imagen\n"
    "Ejemplo: /content/drive/MyDrive/archivo\n"
    "Introduce la ruta de la carpeta dentro de tu Google Drive que contiene:\n",
    title="🧠 Instrucciones"
))


MODEL = input("📦 Ruta del modelo (.h5): ").strip()
TEST_NIFTI = input("🧠 Ruta del NIfTI a analizar (.nii / .nii.gz): ").strip()

IMG_TARGET = 120

# -----------------------------
# VALIDACIONES
# -----------------------------
def archivo_existe_colab(ruta):
    return os.system(f'test -f "{ruta}"') == 0

def validar_archivos(model, test_nifti):
  errores = []

  if not archivo_existe_colab(MODEL):
    errores.append("❌ El modelo no existe")
  elif not MODEL.endswith('.h5'):
    errores.append("❌ El modelo no es un archivo .h5")

  if not archivo_existe_colab(TEST_NIFTI):
    errores.append("❌ El archivo NIfTI no existe")
  elif not TEST_NIFTI.lower().endswith(('.nii', '.nii.gz')):
    errores.append("❌ El archivo no es un NIfTI válido")

  if errores:
    for e in errores:
      print(Panel(e, style="red"))
    raise SystemExit("⛔ Corrige los errores antes de continuar")

try:
  validar_archivos(MODEL, TEST_NIFTI)
  print(Panel("✔ Validación de archivos correcta", style="green"))
except SystemExit as e:
  print(Panel(str(e), title="❌ ERROR DE VALIDACIÓN", style="red"))
except Exception as e:
  print(Panel(str(e), title="❌ ERROR INESPERADO", style="red"))


# Paso 2 - Configuración del Modelo

In [ ]:
#@title 🦾 Configuración del Modelo

import tensorflow as tf
import numpy as np
from keras.models import load_model
from rich import print
from rich.panel import Panel

def dice_coef(y_true, y_pred, smooth=1):
  y_true = tf.cast(y_true, tf.float32)
  y_pred = tf.cast(y_pred, tf.float32)
  intersection = tf.reduce_sum(y_true * y_pred)
  return (2. * intersection + smooth) / (tf.reduce_sum(y_true) + tf.reduce_sum(y_pred) + smooth)


def dice_loss(y_true, y_pred):
  return 1 - dice_coef(y_true, y_pred)


def bce_dice_loss(y_true, y_pred):
  bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
  return bce + dice_loss(y_true, y_pred)

try:
  model = load_model(MODEL, custom_objects={
    'dice_coef': dice_coef,
    'dice_loss': dice_loss,
    'bce_dice_loss': bce_dice_loss
  })
  print(Panel("✔ Modelo cargado correctamente", style="green"))
except Exception as e:
  print(Panel(str(e), title="❌ ERROR AL CARGAR EL MODELO", style="red"))
  raise SystemExit

# Paso 3 - Normalización de NIFTI

⚠️ Usa la MISMA orientación que en el entrenamiento

In [ ]:
#@title 🧬 Carga y validación de NIfTI
import nibabel as nib
import numpy as np
from rich.panel import Panel


try:
  nii = nib.load(TEST_NIFTI)
  vol = nii.get_fdata()
except Exception as e:
  raise RuntimeError(f"❌ Error al cargar el NIfTI: {e}")

# Normalización segura
vmin, vmax = np.min(vol), np.max(vol)
if vmax - vmin == 0:
  raise ValueError("❌ El volumen tiene intensidad constante")


vol = (vol - vmin) / (vmax - vmin)


# Orientación igual al entrenamiento
vol = np.rot90(vol, k=1, axes=(0,1))
vol = np.flip(vol, axis=0)


print(Panel(f"✔ NIfTI cargado correctamente\n Dimensiones: {vol.shape}", style="green"))

max_slice = vol.shape[2] - 1

try:
  slice_inicio = int(input(f"Slice inicial (0–{max_slice}): "))
  slice_fin = int(input(f"Slice final (0–{max_slice}): "))
except ValueError:
  raise ValueError("❌ Debes introducir números enteros")


if slice_inicio < 0 or slice_fin > max_slice:
  raise ValueError("❌ Rango fuera de límites")


if slice_inicio > slice_fin:
  raise ValueError("❌ El slice inicial no puede ser mayor que el final")


print(f"✔ Procesando slices {slice_inicio} → {slice_fin}")

In [ ]:
brain_vol, isq_vol, porcentaje = calcular_volumenes(
    mask_brain_total,
    mask_isq_total,
    voxel_size
)

print("\n=============================")
print("RESULTADOS DEL ESTUDIO")
print("=============================")
print(f"Volumen de cerebro: {brain_vol:.2f} ml")
print(f"Volumen de isquemia: {isq_vol:.2f} ml")
print(f"Porcentaje afectado: {porcentaje:.2f} %")
print("=============================\n")

In [ ]:
# -----------------------------
# Visualización 3D interactiva
# -----------------------------
import plotly.graph_objects as go
import numpy as np

def visualizar_3d_completo(mask_brain, mask_isq, nii, opacity_brain=0.1, opacity_isq=0.3):
    """
    Visualización 3D del cerebro e isquemia con proporciones reales.

    Parámetros:
    - mask_brain: máscara binaria 3D del cerebro
    - mask_isq: máscara binaria 3D de isquemia
    - nii: objeto nibabel del NIfTI original (para obtener voxel size)
    - opacity_brain: opacidad del cerebro
    - opacity_isq: opacidad de la isquemia
    """

    # Tamaño de voxel real
    voxel_size = nii.header.get_zooms()[:3]  # (dx, dy, dz)
    dx, dy, dz = voxel_size
    print(f"Tamaño de voxel: X={dx}mm, Y={dy}mm, Z={dz}mm")

    # Convertir máscaras a booleanas
    brain_bool = mask_brain.astype(bool)
    isq_bool = mask_isq.astype(bool)

    # Crear malla de coordenadas físicas
    X = np.arange(brain_bool.shape[0]) * dx
    Y = np.arange(brain_bool.shape[1]) * dy
    Z = np.arange(brain_bool.shape[2]) * dz

    Xf = X.repeat(brain_bool.shape[1]*brain_bool.shape[2])
    Yf = np.tile(Y.repeat(brain_bool.shape[2]), brain_bool.shape[0])
    Zf = np.tile(Z, brain_bool.shape[0]*brain_bool.shape[1])

    # Crear figura
    fig = go.Figure()

    # Cerebro (verde translúcido)
    fig.add_trace(go.Volume(
        x=Xf,
        y=Yf,
        z=Zf,
        value=brain_bool.flatten().astype(int),
        isomin=0.5,
        isomax=1,
        opacity=opacity_brain,
        surface_count=1,
        colorscale='Greens',
        name='Cerebro'
    ))

    # Isquemia (rojo translúcido)
    fig.add_trace(go.Volume(
        x=Xf,
        y=Yf,
        z=Zf,
        value=isq_bool.flatten().astype(int),
        isomin=0.5,
        isomax=1,
        opacity=opacity_isq,
        surface_count=1,
        colorscale='Reds',
        name='Isquemia'
    ))

    # Configurar escena
    fig.update_layout(
        scene=dict(
            xaxis_title='X (mm)',
            yaxis_title='Y (mm)',
            zaxis_title='Z (mm)',
            aspectmode='data'  # respeta proporciones reales
        ),
        width=900,
        height=700,
        title='Volumen 3D de cerebro e isquemia (escala real)'
    )

    fig.show()

# -----------------------------
# Uso
# -----------------------------
# Solo ejecutar después de haber calculado las máscaras y cargado el NIfTI
# mask_brain_total, mask_isq_total y nii deben existir

visualizar_3d_completo(mask_brain_total, mask_isq_total, nii)


In [ ]:
from PIL import Image
import os
import zipfile

def exportar_rois_compatible(mask_brain, mask_isq, slice_inicio, slice_fin, output_dir="output_roi"):
    """
    Exporta cada slice como imagen binaria 8-bit para convertir a .roi en ImageJ.
    Luego comprime todo en un zip.

    - mask_brain, mask_isq: 3D numpy binarios (0/1)
    - slice_inicio, slice_fin: rango de slices procesadas
    """
    os.makedirs(output_dir, exist_ok=True)

    brain_dir = os.path.join(output_dir, "cerebro")
    isq_dir   = os.path.join(output_dir, "isquemia")
    os.makedirs(brain_dir, exist_ok=True)
    os.makedirs(isq_dir, exist_ok=True)

    for i in range(slice_inicio, slice_fin + 1):
        idx = str(i+1).zfill(4)

        # Cerebro
        brain_slice = (mask_brain[:, :, i] * 255).astype('uint8')
        brain_img = Image.fromarray(brain_slice)
        brain_img.save(os.path.join(brain_dir, f"cerebro_slice_{idx}.tif"))

        # Isquemia
        isq_slice = (mask_isq[:, :, i] * 255).astype('uint8')
        isq_img = Image.fromarray(isq_slice)
        isq_img.save(os.path.join(isq_dir, f"isquemia_slice_{idx}.tif"))

    # Comprimir
    zip_path = os.path.join(output_dir, "rois_tif.zip")
    with zipfile.ZipFile(zip_path, 'w') as zipf:
        for f in os.listdir(brain_dir):
            zipf.write(os.path.join(brain_dir, f), arcname=f"cerebro/{f}")
        for f in os.listdir(isq_dir):
            zipf.write(os.path.join(isq_dir, f), arcname=f"isquemia/{f}")

    print(f"✅ Exportación completada. Zip generado en: {zip_path}")
    print("⚠ Para usar en ROI Manager: abrir cada TIFF en ImageJ y usar 'Image > Adjust > Threshold' > 'Analyze > Tools > ROI Manager > Add'")

exportar_rois_compatible(mask_brain_total, mask_isq_total, slice_inicio, slice_fin)
